# Sparse-view CT reconstruction with a U-Net

A phase-contrast micro-CT scan of a small tissue sample can take many hours because scan time grows with the number of projection angles. Fewer angles give a shorter scan, but filtered back projection (FBP) leaves streak artifacts and fine details are lost first.

This notebook tests on simulated data how much a small U-Net can recover. The workflow makes synthetic phantoms, simulates noisy projections, reconstructs at 180 and 30 angles, trains the network to map the sparse reconstruction to the 180-angle reference, and evaluates PSNR and SSIM.

## 1. Phantoms

The data are synthetic: a soft-tissue body, elliptical structures with different attenuation, and small bright dots representing fine high-contrast detail.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import radon, iradon
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

os.makedirs('outputs', exist_ok=True)
rng = np.random.default_rng(0)

def make_phantom(size):
    yy, xx = np.mgrid[0:size, 0:size]
    yy = (yy - size / 2) / (size / 2)
    xx = (xx - size / 2) / (size / 2)
    img = np.zeros((size, size), dtype=np.float32)
    body = (xx / 0.85) ** 2 + (yy / 0.95) ** 2 <= 1.0
    img[body] = 0.25
    for _ in range(rng.integers(3, 7)):
        cx, cy = rng.uniform(-0.5, 0.5, 2)
        ax, ay = rng.uniform(0.10, 0.40, 2)
        ang = rng.uniform(0, np.pi)
        xr = (xx - cx) * np.cos(ang) + (yy - cy) * np.sin(ang)
        yr = -(xx - cx) * np.sin(ang) + (yy - cy) * np.cos(ang)
        mask = (xr / ax) ** 2 + (yr / ay) ** 2 <= 1.0
        img[mask & body] += rng.uniform(-0.15, 0.35)
    for _ in range(rng.integers(0, 6)):
        cx, cy = rng.uniform(-0.6, 0.6, 2)
        r = rng.uniform(0.01, 0.03)
        dot = (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
        img[dot & body] = 1.0
    return img

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a in ax:
    a.imshow(make_phantom(128), cmap='gray'); a.axis('off')
fig.tight_layout(); fig.savefig('outputs/fig_phantoms.png', dpi=150); plt.show()

## 2. Simulating the scan and reconstruction

The Radon transform produces the projections. Photon-counting noise is simulated with a Poisson distribution after Beer-Lambert attenuation, then the logarithm gives a noisy sinogram. FBP with a ramp filter reconstructs the image.

In [ ]:
def simulate_scan(img, n_angles, photons=2e4):
    angles = np.linspace(0.0, 180.0, n_angles, endpoint=False)
    sino = radon(img, theta=angles, circle=True)
    scale = 3.0 / sino.max()
    counts = rng.poisson(photons * np.exp(-sino * scale))
    counts = np.clip(counts, 1, None)
    sino = -np.log(counts / photons) / scale
    rec = iradon(sino, theta=angles, circle=True, filter_name='ramp')
    return sino, rec.astype(np.float32)

In [ ]:
img = make_phantom(128)
_, ref = simulate_scan(img, 180)
_, fbp = simulate_scan(img, 30)
fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
for a, im, t in zip(ax, [img, ref, fbp], ['phantom', 'FBP, 180 views', 'FBP, 30 views']):
    a.imshow(im, cmap='gray'); a.set_title(t); a.axis('off')
fig.tight_layout(); fig.savefig('outputs/fig_simulation.png', dpi=150); plt.show()

## 3. Dataset and metrics

600 simulated image pairs are generated once and stored. The 180-view FBP is the supervised reference; it is not the original phantom.

In [ ]:
N_IMAGES, SIZE, FULL, SPARSE = 600, 128, 180, 30
truth, ref_list, fbp_list = [], [], []
for i in range(N_IMAGES):
    img = make_phantom(SIZE)
    _, ref = simulate_scan(img, FULL)
    _, fbp = simulate_scan(img, SPARSE)
    truth.append(img); ref_list.append(ref); fbp_list.append(fbp)
    if (i + 1) % 50 == 0: print(f'{i + 1}/{N_IMAGES}')
np.savez_compressed('outputs/dataset.npz', truth=np.stack(truth), ref=np.stack(ref_list), fbp=np.stack(fbp_list), full=FULL, sparse=SPARSE)

In [ ]:
def scores(pred, ref):
    rng_val = float(ref.max() - ref.min()) or 1.0
    return peak_signal_noise_ratio(ref, pred, data_range=rng_val), structural_similarity(ref, pred, data_range=rng_val)
def mean_scores(preds, refs):
    vals = np.array([scores(p, r) for p, r in zip(preds, refs)])
    return vals[:,0].mean(), vals[:,1].mean()
d = np.load('outputs/dataset.npz')
print(d['fbp'].shape)
print('baseline, sparse FBP vs reference:', mean_scores(d['fbp'][:20], d['ref'][:20]))

## 4. Splits and the network

The data are split 80/10/10. The test set is held back until the end. The U-Net has two downsampling stages, a bottleneck, and two upsampling stages with skip connections.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
ref_all = d['ref'].astype(np.float32); fbp_all = d['fbp'].astype(np.float32)
indices = np.random.default_rng(42).permutation(len(fbp_all))
n_train, n_val = int(0.8*len(indices)), int(0.1*len(indices))
train_idx = indices[:n_train]; val_idx = indices[n_train:n_train+n_val]; test_idx = indices[n_train+n_val:]
X_train, Y_train = fbp_all[train_idx], ref_all[train_idx]
X_val, Y_val = fbp_all[val_idx], ref_all[val_idx]
X_test, Y_test = fbp_all[test_idx], ref_all[test_idx]
def to_tensor(x): return torch.from_numpy(x[:, None, :, :])
train_loader = DataLoader(TensorDataset(to_tensor(X_train), to_tensor(Y_train)), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(to_tensor(X_val), to_tensor(Y_val)), batch_size=16, shuffle=False)
test_loader = DataLoader(TensorDataset(to_tensor(X_test), to_tensor(Y_test)), batch_size=16, shuffle=False)

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.Conv2d(in_ch,out_ch,3,padding=1), nn.ReLU(inplace=True), nn.Conv2d(out_ch,out_ch,3,padding=1), nn.ReLU(inplace=True))
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch): super().__init__(); self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch,out_ch))
    def forward(self, x): return self.net(x)

class Bottleneck(nn.Module):
    def __init__(self, in_ch, out_ch): super().__init__(); self.net = DoubleConv(in_ch,out_ch)
    def forward(self, x): return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__(); self.up = nn.ConvTranspose2d(in_ch,in_ch//2,2,stride=2); self.conv = DoubleConv(in_ch//2+skip_ch,out_ch)
    def forward(self, x1, x2):
        x1 = self.up(x1); x = torch.cat([x2,x1], dim=1); return self.conv(x)

class UNet(nn.Module):
    def __init__(self, n_channels=1, n_classes=1):
        super().__init__(); self.inc=DoubleConv(n_channels,64); self.down1=Down(64,128); self.down2=Down(128,256); self.bottleneck=Bottleneck(256,512); self.up1=Up(512,128,256); self.up2=Up(256,64,128); self.outc=nn.Conv2d(128,n_classes,1)
    def forward(self, x):
        x1=self.inc(x); x2=self.down1(x1); x3=self.down2(x2); x4=self.bottleneck(x3); x=self.up1(x4,x2); x=self.up2(x,x1); return self.outc(x)

In [ ]:
model = UNet().to(device)
print('parameters:', sum(p.numel() for p in model.parameters()) / 1e6, 'M')
sample_x, _ = next(iter(train_loader))
with torch.no_grad(): sample_out = model(sample_x.to(device))
print('input :', tuple(sample_x.shape)); print('output:', tuple(sample_out.shape))

## 5. Training

Training and validation loss are recorded. The checkpoint with the lowest validation loss is saved.

In [ ]:
criterion = nn.MSELoss(); optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 10; history = []; best_val = float('inf')
for epoch in range(EPOCHS):
    model.train(); train_loss = 0.0
    for x,y in train_loader:
        x,y=x.to(device),y.to(device); optimizer.zero_grad(); loss=criterion(model(x),y); loss.backward(); optimizer.step(); train_loss += loss.item()
    train_loss /= len(train_loader); model.eval(); val_loss=0.0
    with torch.no_grad():
        for x,y in val_loader: val_loss += criterion(model(x.to(device)),y.to(device)).item()
    val_loss /= len(val_loader); history.append((train_loss,val_loss))
    if val_loss < best_val:
        best_val = val_loss; torch.save(model.state_dict(),'outputs/unet_model.pth')
    print(f'epoch {epoch+1:2d}/{EPOCHS} | train {train_loss:.6f} | val {val_loss:.6f}')
print('best validation loss:', best_val)

In [ ]:
h=np.array(history); fig,ax=plt.subplots(figsize=(5,3.2)); ax.plot(np.arange(1,len(h)+1),h[:,0],label='train'); ax.plot(np.arange(1,len(h)+1),h[:,1],label='validation'); ax.set_xlabel('epoch'); ax.set_ylabel('MSE loss'); ax.legend(); fig.tight_layout(); fig.savefig('outputs/fig_loss.png',dpi=150); plt.show()

## 6. Test set

The best checkpoint is loaded and evaluated on the held-out test set against the same 180-angle reference.

In [ ]:
model.load_state_dict(torch.load('outputs/unet_model.pth', map_location=device)); model.eval(); test_preds=[]
with torch.no_grad():
    for x,_ in test_loader: test_preds.append(model(x.to(device)).cpu().numpy())
test_preds=np.concatenate(test_preds,axis=0)[:,0,:,:]
fbp_psnr,fbp_ssim=mean_scores(X_test,Y_test); unet_psnr,unet_ssim=mean_scores(test_preds,Y_test)
print(f'test set: {len(X_test)} images, {SPARSE} of {FULL} views')
print(f"{'method':<18}{'PSNR (dB)':>12}{'SSIM':>10}")
print(f"{'sparse-view FBP':<18}{fbp_psnr:>12.2f}{fbp_ssim:>10.3f}")
print(f"{'U-Net':<18}{unet_psnr:>12.2f}{unet_ssim:>10.3f}")
with open('outputs/results.md','w') as f:
    f.write('| Method | PSNR (dB) | SSIM |\\n|---|---:|---:|\\n')
    f.write(f'| Sparse-view FBP | {fbp_psnr:.2f} | {fbp_ssim:.3f} |\\n| U-Net | {unet_psnr:.2f} | {unet_ssim:.3f} |\\n')

In [ ]:
n_show=3; fig,axes=plt.subplots(n_show,3,figsize=(9.5,9.5))
for i in range(n_show):
    p_fbp,s_fbp=scores(X_test[i],Y_test[i]); p_net,s_net=scores(test_preds[i],Y_test[i])
    panels=[(X_test[i],f'FBP, {SPARSE} views\n{p_fbp:.1f} dB / SSIM {s_fbp:.2f}'),(test_preds[i],f'U-Net\n{p_net:.1f} dB / SSIM {s_net:.2f}'),(Y_test[i],f'reference, {FULL} views')]
    for ax,(im,title) in zip(axes[i],panels): ax.imshow(im,cmap='gray'); ax.set_title(title,fontsize=9); ax.axis('off')
fig.tight_layout(); fig.savefig('outputs/fig_comparison.png',dpi=150); plt.show()